In [1]:
import pandas as pd


In [16]:
df = pd.read_csv("data/train_test.csv")
print(df.shape)
print(df.columns.tolist())

(48000, 14)
['load_id', 'pickup', 'delivery', 'pickup_lat', 'pickup_lon', 'delivery_lat', 'delivery_lon', 'distance', 'equipment', 'weight', 'date', 'market_index', 'quote_signal', 'posted_rate']


In [17]:
print(df.dtypes)
print(df.isna().sum())

load_id          object
pickup           object
delivery         object
pickup_lat      float64
pickup_lon      float64
delivery_lat    float64
delivery_lon    float64
distance        float64
equipment        object
weight          float64
date             object
market_index    float64
quote_signal    float64
posted_rate     float64
dtype: object
load_id           0
pickup            0
delivery          0
pickup_lat        0
pickup_lon        0
delivery_lat      0
delivery_lon      0
distance          0
equipment         0
weight          300
date              0
market_index    374
quote_signal      0
posted_rate       0
dtype: int64


In [18]:
df["date"] = pd.to_datetime(df["date"])
print("Train date range:", df["date"].min(), "to", df["date"].max())

print("Negative weights:", (df["weight"] < 0).sum())
print("Negative distances:", (df["distance"] < 0).sum())
print("Negative rates:", (df["posted_rate"] < 0).sum())

print(df["equipment"].value_counts())

Train date range: 2025-01-01 00:00:00 to 2025-10-31 00:00:00
Negative weights: 292
Negative distances: 0
Negative rates: 0
equipment
Dry Van    27202
Reefer     12045
Flatbed     8753
Name: count, dtype: int64


In [19]:
val = pd.read_csv("data/validation.csv")


In [20]:
val["date"] = pd.to_datetime(val["date"])
print("Validation date range:", val["date"].min(), "to", val["date"].max())
print(val.shape)

Validation date range: 2025-11-01 00:00:00 to 2025-12-31 00:00:00
(12000, 13)


In [21]:
holdout_start = pd.to_datetime("2025-09-01")
train_part = df[df["date"] < holdout_start]
holdout_part = df[df["date"] >= holdout_start]
print("Train rows:", len(train_part))
print("Holdout rows:", len(holdout_part))

Train rows: 38477
Holdout rows: 9523


In [ ]:
train_pickups = set(df["pickup"])
val_pickups = set(val["pickup"])
print("Cities in validation NOT seen in training:", val_pickups - train_pickups)


Cities in validation NOT seen in training: {'Chicago', 'Norfolk', 'Knoxville', 'Laredo', 'Jackson', 'Allentown', 'San Diego', 'Charlotte'}


In [23]:
train_lanes = set(zip(df["pickup"], df["delivery"]))
val_lanes = set(zip(val["pickup"], val["delivery"]))
print("Lanes in validation NOT seen in training:", len(val_lanes - train_lanes), "out of", len(val_lanes))

Lanes in validation NOT seen in training: 736 out of 4214


In [24]:
print(df.corr(numeric_only=True)["posted_rate"].sort_values(ascending=False))

posted_rate     1.000000
distance        0.908519
weight          0.034840
market_index    0.034165
quote_signal   -0.039858
pickup_lat     -0.090873
delivery_lat   -0.091970
pickup_lon     -0.255058
delivery_lon   -0.257086
Name: posted_rate, dtype: float64


In [25]:
df["rate_per_mile"] = df["posted_rate"] / df["distance"]
print(df.groupby("equipment")["rate_per_mile"].mean())


equipment
Dry Van    2.115401
Flatbed    2.294729
Reefer     2.383376
Name: rate_per_mile, dtype: float64


In [26]:
df["month"] = df["date"].dt.month
print(df.groupby("month")["rate_per_mile"].mean())

month
1     2.100221
2     2.122885
3     2.206132
4     2.210309
5     2.260106
6     2.332484
7     2.255952
8     2.188584
9     2.229725
10    2.239763
Name: rate_per_mile, dtype: float64


In [27]:
df["weight"] = df["weight"].abs()
print("Negative weights after fix:", (df["weight"] < 0).sum())
print(df["weight"].describe())

Negative weights after fix: 0
count    47700.000000
mean     31417.249245
std       7996.515399
min       5000.000000
25%      25922.000000
50%      31496.000000
75%      37063.250000
max      47500.000000
Name: weight, dtype: float64


In [28]:
weight_median = df["weight"].median()
market_index_median = df["market_index"].median()
print("Weight median:", weight_median)
print("Market index median:", market_index_median)

df["weight"] = df["weight"].fillna(weight_median)
df["market_index"] = df["market_index"].fillna(market_index_median)

print("Missing after fill:")
print(df[["weight", "market_index"]].isna().sum())

Weight median: 31496.0
Market index median: 1.0558
Missing after fill:
weight          0
market_index    0
dtype: int64


In [29]:
import numpy as np

def haversine(lat1, lon1, lat2, lon2):
    R = 3958.8  # miles
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1)*np.cos(lat2)*np.sin(dlon/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

df["haversine_dist"] = haversine(df["pickup_lat"], df["pickup_lon"], df["delivery_lat"], df["delivery_lon"])
df["dist_ratio"] = df["distance"] / df["haversine_dist"]
print(df["dist_ratio"].describe())
print(df.sort_values("dist_ratio", ascending=False)[["pickup","delivery","distance","haversine_dist","dist_ratio"]].head(5))

count    48000.000000
mean         1.194681
std          0.143541
min          1.058090
25%          1.164765
50%          1.182231
75%          1.204765
max          9.634919
Name: dist_ratio, dtype: float64
            pickup     delivery  distance  haversine_dist  dist_ratio
9348    Shreveport  New Orleans      70.0         7.26524    9.634919
37183   Shreveport  New Orleans      70.0         7.26524    9.634919
12480  New Orleans   Shreveport      70.0         7.26524    9.634919
34136   Shreveport  New Orleans      70.0         7.26524    9.634919
38469   Shreveport  New Orleans      70.0         7.26524    9.634919
